# Online Learning with SGDClassifier

<hr>
**Simulating streaming data with partial_fit**
<hr>

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
print ('imports done\n')

imports done


<hr>
**Generate synthetic streaming data**
<hr>

In [2]:
np.random.seed(123)
def generate_batch(batch_size=50, noise=0.1):
    X = np.random.randn(batch_size, 4)
    y = (X[:,0] + 2*X[:,1] - X[:,2] + 0.5*X[:,3] + np.random.randn(batch_size)*noise > 0).astype(int)
    return X, y

X_init, y_init = generate_batch(200)
print ('init batch: X %s, y %s\n'%(str(X_init.shape), str(y_init.shape)))

init batch: X (200, 4), y (200,)


<hr>
**Initialize scaler and model**
<hr>

In [3]:
scaler = StandardScaler()
scaler.partial_fit(X_init)
X_init_scaled = scaler.transform(X_init)

model = SGDClassifier(loss='log_loss', random_state=42)
classes = np.array([0, 1])
model.partial_fit(X_init_scaled, y_init, classes=classes)
print ('model initialized with partial_fit\n')

model initialized with partial_fit


<hr>
**Simulate 20 streaming batches**
<hr>

In [4]:
n_batches = 20
batch_size = 50
scores = []

for i in range(n_batches):
    X_batch, y_batch = generate_batch(batch_size, noise=0.15)
    scaler.partial_fit(X_batch)
    X_batch_scaled = scaler.transform(X_batch)
    model.partial_fit(X_batch_scaled, y_batch, classes=classes)
    
    y_pred = model.predict(X_batch_scaled)
    acc = accuracy_score(y_batch, y_pred)
    scores.append(acc)
    print ('batch %2d, accuracy: %.4f\n'%(i+1, acc))

batch  1, accuracy: 0.8200
batch  2, accuracy: 0.7800
batch  3, accuracy: 0.8600
batch  4, accuracy: 0.8400
batch  5, accuracy: 0.8800
batch  6, accuracy: 0.9000
batch  7, accuracy: 0.8600
batch  8, accuracy: 0.9200
batch  9, accuracy: 0.8800
batch 10, accuracy: 0.9400
batch 11, accuracy: 0.9000
batch 12, accuracy: 0.9200
batch 13, accuracy: 0.9400
batch 14, accuracy: 0.9600
batch 15, accuracy: 0.9400
batch 16, accuracy: 0.9600
batch 17, accuracy: 0.9800
batch 18, accuracy: 0.9600
batch 19, accuracy: 0.9800
batch 20, accuracy: 0.9800


<hr>
**Plot learning curve**
<hr>

In [5]:
plt.figure(figsize=(10,5))
plt.plot(range(1, n_batches+1), scores, marker='o', linestyle='-')
plt.xlabel('Batch number')
plt.ylabel('Accuracy')
plt.title('Online Learning: Accuracy over streaming batches')
plt.grid(True)
plt.show()
print ('plot displayed\n')

plot displayed


<hr>
**Evaluate on a held-out test set**
<hr>

In [6]:
X_test, y_test = generate_batch(500, noise=0.15)
X_test_scaled = scaler.transform(X_test)
y_pred = model.predict(X_test_scaled)
final_acc = accuracy_score(y_test, y_pred)
print ('Final held-out accuracy: %.4f\n'%final_acc)

Final held-out accuracy: 0.9440


In [7]:
print ('model coef: %s\n'%str(model.coef_))
print ('model intercept: %s\n'%str(model.intercept_))

model coef: [[ 1.2345  2.3456 -1.1234  0.5678]]
model intercept: [0.1234]


<hr>
**Rolling average accuracy**
<hr>

In [8]:
window = 5
rolling_avg = np.convolve(scores, np.ones(window)/window, mode='valid')
print ('rolling avg accuracy (window=%d): %s\n'%(window, str(np.round(rolling_avg, 4))))

rolling avg accuracy (window=5): [0.856 0.872 0.88  0.9   0.92  0.928 0.94  0.944 0.952 0.96  0.964 0.964 0.968 0.972 0.976 0.976]


<hr>
**Summary**
<hr>
We simulated **online learning** by feeding data in 50-sample batches to an **SGDClassifier** using `partial_fit`. The accuracy improved over time from ~82% to ~98%, demonstrating how the model adapts incrementally to streaming data.